In [15]:
import cv2
import mediapipe as mp
import numpy as np

def overlay_png(background, overlay, x, y):
    bh, bw = background.shape[:2]
    oh, ow = overlay.shape[:2]

    # 예외처리 : x,y시작점이 화면 아예 밖에있거나, 오버레이가 화면안으로 안들어오면은 원본 반
    if x >= bw or y >= bh or x + ow <= 0 or y + oh <= 0:
        return background

    # 잘라낼 배경 영역 계산
    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + ow, bw)
    y2 = min(y + oh, bh)

    # 오버레이 영역 계산, 배경 영역과 값이 같아야함 
    ox1 = max(0, -x)
    oy1 = max(0, -y)
    ox2 = ox1 + (x2 - x1)
    oy2 = oy1 + (y2 - y1)

    roi = background[y1:y2, x1:x2]
    overlay_crop = overlay[oy1:oy2, ox1:ox2]

    if overlay_crop is None or overlay_crop.shape[2] != 4:
        return background

    overlay_rgb = overlay_crop[:, :, :3]
    alpha = overlay_crop[:, :, 3:4] / 255.0

    background[y1:y2, x1:x2] = (overlay_rgb * alpha + roi * (1 - alpha)).astype(np.uint8)
    return background


cap = cv2.VideoCapture('threeman_talk.mp4')

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

sunglasses = cv2.imread('sunglasses.png', cv2.IMREAD_UNCHANGED)
print("sunglasses shape:", None if sunglasses is None else sunglasses.shape)

if sunglasses is None:
    raise Exception("sunglasses.png 못 읽음")

if len(sunglasses.shape) != 3 or sunglasses.shape[2] != 4:
    raise Exception("알파채널 있는 PNG 아님")

with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5
) as face_detection:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        image.flags.writeable = False
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb)

        image.flags.writeable = True

        h, w, _ = image.shape

        if results.detections:
            for detection in results.detections:
                kps = detection.location_data.relative_keypoints

                right_eye = (int(kps[0].x * w), int(kps[0].y * h))
                left_eye  = (int(kps[1].x * w), int(kps[1].y * h))

                eye_distance = int(np.hypot(
                    left_eye[0] - right_eye[0],
                    left_eye[1] - right_eye[1]
                ))

                if eye_distance < 10:
                    continue

                center_x = (right_eye[0] + left_eye[0]) // 2
                center_y = (right_eye[1] + left_eye[1]) // 2

                sh, sw = sunglasses.shape[:2]
                glass_width = int(eye_distance * 2.5)
                glass_height = int(glass_width * sh / sw)

                resized = cv2.resize(sunglasses, (glass_width, glass_height))

                x = center_x - glass_width // 2
                y = center_y - glass_height // 2

                # 처음엔 보정 적게
                y = y + 10

                # 디버깅용
                # cv2.circle(image, right_eye, 5, (0, 0, 255), cv2.FILLED)
                # cv2.circle(image, left_eye, 5, (255, 0, 0), cv2.FILLED)
                # cv2.rectangle(image, (x, y), (x + glass_width, y + glass_height), (0, 255, 0), 2)

                image = overlay_png(image, resized, x, y)

        cv2.imshow('Sunglasses Filter', image)

        if cv2.waitKey(50) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

sunglasses shape: (1201, 1201, 4)


In [17]:
import cv2
import mediapipe as mp
import numpy as np

#붙이기 전용 함수 
def overlay_png(background, overlay, x, y):
    bh, bw = background.shape[:2]
    oh, ow = overlay.shape[:2]

    if x >= bw or y >= bh or x + ow <= 0 or y + oh <= 0:
        return background

    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + ow, bw)
    y2 = min(y + oh, bh)

    ox1 = max(0, -x)
    oy1 = max(0, -y)
    ox2 = ox1 + (x2 - x1)
    oy2 = oy1 + (y2 - y1)

    roi = background[y1:y2, x1:x2]
    overlay_crop = overlay[oy1:oy2, ox1:ox2]

    if overlay_crop is None or overlay_crop.shape[2] != 4:
        return background

    overlay_rgb = overlay_crop[:, :, :3]
    alpha = overlay_crop[:, :, 3:4] / 255.0

    background[y1:y2, x1:x2] = (overlay_rgb * alpha + roi * (1 - alpha)).astype(np.uint8)
    return background


cap = cv2.VideoCapture('yooju.mp4')

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

sunglasses = cv2.imread('sunglass2.png', cv2.IMREAD_UNCHANGED)

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:

    while cap.isOpened():
        ret, image = cap.read()
        if not ret:
            break

        image.flags.writeable = False
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb)

        image.flags.writeable = True

        h, w, _ = image.shape

        if results.detections:
            for detection in results.detections:
                kps = detection.location_data.relative_keypoints

                right_eye = (int(kps[0].x * w), int(kps[0].y * h))
                left_eye  = (int(kps[1].x * w), int(kps[1].y * h))

                eye_distance = int(np.hypot(
                    left_eye[0] - right_eye[0],
                    left_eye[1] - right_eye[1]
                ))

                if eye_distance < 10:
                    continue

                center_x = (right_eye[0] + left_eye[0]) // 2
                center_y = (right_eye[1] + left_eye[1]) // 2

                sh, sw = sunglasses.shape[:2]
                glass_width = int(eye_distance * 2.5)
                glass_height = int(glass_width * sh / sw)

                resized = cv2.resize(sunglasses, (glass_width, glass_height))

                x = center_x - glass_width // 2
                y = center_y - glass_height // 2

                # 처음엔 보정 적게
                y = y + 10

                # 디버깅용
                # cv2.circle(image, right_eye, 5, (0, 0, 255), cv2.FILLED)
                # cv2.circle(image, left_eye, 5, (255, 0, 0), cv2.FILLED)
                # cv2.rectangle(image, (x, y), (x + glass_width, y + glass_height), (0, 255, 0), 2)

                image = overlay_png(image, resized, x, y)

        cv2.imshow('Sunglasses Filter', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [13]:
import cv2
import mediapipe as mp

# -----------------------------
# MediaPipe 준비
# -----------------------------
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# -----------------------------
# 영상, PNG 이미지 읽기
# -----------------------------
cap = cv2.VideoCapture('yooju.mp4')

glasses = cv2.imread('sunglass2.png', cv2.IMREAD_UNCHANGED)
glasses = cv2.resize(glasses, (350, 350))

# beard = cv2.imread('santa.png', cv2.IMREAD_UNCHANGED)
# beard = cv2.resize(beard, (300, 300))

# -----------------------------
# 얼굴 검출 시작
# -----------------------------
with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 현재 프레임 크기
        frame_h, frame_w, _ = frame.shape

        # MediaPipe 처리용으로 BGR -> RGB
        frame.flags.writeable = False
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb_frame)

        # 다시 그릴 수 있게 변경
        frame.flags.writeable = True
        frame = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR)

        if results.detections:
            for detection in results.detections:
                # 얼굴 박스, 키포인트 표시
                mp_drawing.draw_detection(frame, detection)

                # -----------------------------
                # 1. 안경 붙이기
                # -----------------------------
                eye_point = detection.location_data.relative_keypoints[0]

                eye_x = int(eye_point.x * frame_w)
                eye_y = int(eye_point.y * frame_h)

                glasses_h, glasses_w, _ = glasses.shape

                # 안경이 화면 안에 들어올 때만 합성
                if (
                    0 <= eye_y - glasses_h // 2 and
                    eye_y + glasses_h // 2 <= frame_h and
                    0 <= eye_x - glasses_w // 2 and
                    eye_x + glasses_w // 2 <= frame_w
                ):
                    y1 = eye_y - glasses_h // 2
                    y2 = eye_y + glasses_h // 2
                    x1 = eye_x - glasses_w // 2
                    x2 = eye_x + glasses_w // 2

                    roi = frame[y1:y2, x1:x2]

                    alpha = glasses[:, :, 3] / 255.0
                    alpha = alpha[:, :, None]

                    roi[:] = (1 - alpha) * roi + alpha * glasses[:, :, :3]

                

        cv2.imshow('yooju', frame)

        if cv2.waitKey(30) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()